# run — fog + glass_blur Fix & Rerun
**AAAI 2027 — BMIA ImageNet-C**
Fixes scipy API issue, diagnoses both corruptions, then runs full experiment.
Fork from run so ImageNet dataset is already attached.

In [ ]:

# ── Cell 1: Install + ALL fixes (MUST run before imagecorruptions import) ─
import subprocess
subprocess.run(['pip', 'install', 'imagecorruptions', 'scipy', '-q'], check=True)

import numpy as np
import scipy, scipy.ndimage, sys

# FIX 1 — NumPy 2.0 removed np.float_ → fog corruption breaks
if not hasattr(np, 'float_'):
    np.float_ = np.float64
    print('[FIX 1 APPLIED] np.float_ = np.float64')
else:
    print('[OK] np.float_ exists')

# FIX 2 — scikit-image removed multichannel param → glass_blur breaks
from skimage import filters as _skfilters
_orig_gaussian = _skfilters.gaussian

def _patched_gaussian(image, sigma=1, multichannel=None, channel_axis=None, **kwargs):
    if multichannel is not None and channel_axis is None:
        channel_axis = -1 if multichannel else None
    return _orig_gaussian(image, sigma=sigma, channel_axis=channel_axis, **kwargs)

_skfilters.gaussian = _patched_gaussian
import skimage.filters
skimage.filters.gaussian = _patched_gaussian
print('[FIX 2 APPLIED] skimage.filters.gaussian patched (multichannel removed)')

# FIX 3 — scipy.ndimage.filters removed in newer scipy
if not hasattr(scipy.ndimage, 'filters'):
    scipy.ndimage.filters = scipy.ndimage
    print('[FIX 3 APPLIED] scipy.ndimage.filters patched')
if 'scipy.ndimage.filters' not in sys.modules:
    sys.modules['scipy.ndimage.filters'] = scipy.ndimage
    print('[FIX 3 APPLIED] scipy.ndimage.filters injected into sys.modules')

print()
print('numpy  :', np.__version__)
print('scipy  :', scipy.__version__)
import skimage; print('skimage:', skimage.__version__)
print('All fixes applied.')


In [ ]:
# ── Cell 2: Imports ──────────────────────────────────────────────────────
import os, csv, time, traceback, warnings
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from imagecorruptions import corrupt

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Torch  :', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
print('Device :', device)

In [ ]:
# ── Cell 3: Auto-discover dataset paths ─────────────────────────────────
VAL_DIR    = '/kaggle/input/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/val'
LABEL_FILE = '/kaggle/input/imagenet-object-localization-challenge/LOC_val_solution.csv'

# Auto-discover VAL_DIR
if not os.path.isdir(VAL_DIR):
    print('[WARN] Default val path not found. Searching...')
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'val' in dirs:
            candidate = os.path.join(root, 'val')
            if any(f.endswith('.JPEG') for f in os.listdir(candidate)):
                VAL_DIR = candidate
                print(f'[OK] Found val dir: {VAL_DIR}')
                break
    else:
        raise FileNotFoundError(
            'Val directory not found.\n'
            'In Kaggle: Data panel → + Add Data → ImageNet Object Localization Challenge')
else:
    print(f'[OK] Val dir: {VAL_DIR}')

# Auto-discover LABEL_FILE
if not os.path.exists(LABEL_FILE):
    print('[WARN] LOC_val_solution.csv not found. Searching...')
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'LOC_val_solution.csv' in files:
            LABEL_FILE = os.path.join(root, 'LOC_val_solution.csv')
            print(f'[OK] Found label file: {LABEL_FILE}')
            break
    else:
        raise FileNotFoundError(
            'LOC_val_solution.csv not found.\n'
            'In Kaggle: Data panel → + Add Data → ImageNet Object Localization Challenge')
else:
    print(f'[OK] Label file: {LABEL_FILE}')

In [ ]:
# ── Cell 4: Diagnose fog + glass_blur on ONE image ───────────────────────
RESIZE_CROP = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])

sample_path = next(
    os.path.join(VAL_DIR, f)
    for f in sorted(os.listdir(VAL_DIR))
    if f.endswith('.JPEG')
)
print(f'Test image: {sample_path}')

img    = Image.open(sample_path).convert('RGB')
img    = RESIZE_CROP(img)
img_np = np.array(img, dtype=np.uint8)
print(f'Shape: {img_np.shape}  min={img_np.min()}  max={img_np.max()}')

print('\n── Corruption Diagnostic ──')
all_ok = True
for name in ['fog', 'glass_blur']:
    print(f'\n  {name}:')
    try:
        result = corrupt(img_np.copy(), corruption_name=name, severity=5)
        change = np.abs(result.astype(int) - img_np.astype(int)).mean()
        if change > 1.0:
            print(f'    [OK] pixel_change = {change:.2f} — corruption working correctly')
        else:
            print(f'    [BROKEN] pixel_change = {change:.2f} — no effect on image')
            all_ok = False
    except Exception as e:
        print(f'    [ERROR] {type(e).__name__}: {e}')
        traceback.print_exc()
        all_ok = False

print()
if all_ok:
    print('Both corruptions working. Safe to run full experiment (Cell 5 onward).')
else:
    print('One or more corruptions failed. Do NOT run cells below — report the error above.')

In [ ]:
# ── Cell 5: Configuration (same as run) ──────────────────────────────────
SEVERITY    = 5
BATCH_SIZE  = 64
SEED        = 42
NUM_WORKERS = 2
K           = 1000
IMAGES_PER_CORRUPTION = 5000
LEARNING_RATES = [0.005, 0.05]
CORRUPTIONS_TO_RUN = ['fog', 'glass_blur']
RESULTS_DIR = '/kaggle/working/results_fog_glass'

# EATA
E0            = 0.4 * np.log(K)   # 2.763 nats
FISHER_WEIGHT = 2000.0
FISHER_N      = 500

# BMIA
LAMBDA_FIXED = 1.0
LAMBDA_INIT  = 0.5
LAMBDA_MIN   = 0.01
LAMBDA_MAX   = 10.0
TAU          = 0.10

# Collapse detection
COLLAPSE_DOM_THRESH    = 2.0 / K
COLLAPSE_ACTIVE_THRESH = K // 2
COLLAPSE_HARD_THRESH   = K // 5

np.random.seed(SEED)
torch.manual_seed(SEED)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Config ready.')
print(f'E0={E0:.3f}  TAU={TAU}  LAMBDA_FIXED={LAMBDA_FIXED}')
print(f'Corruptions: {CORRUPTIONS_TO_RUN}')

In [ ]:
# ── Cell 6: Build label map + subset (same seed/logic as run) ────────────
import csv as _csv

synsets, rows = set(), {}
with open(LABEL_FILE, 'r') as f:
    for row in _csv.DictReader(f):
        img_id = row['ImageId']
        synset = row['PredictionString'].split()[0]
        synsets.add(synset)
        rows[img_id] = synset

synset_to_idx = {s: i for i, s in enumerate(sorted(synsets))}

img_list = [
    (os.path.join(VAL_DIR, img_id + '.JPEG'), synset_to_idx[syn])
    for img_id, syn in sorted(rows.items())
    if os.path.exists(os.path.join(VAL_DIR, img_id + '.JPEG'))
]

# MUST use same seed and same logic as run to get identical subset
np.random.seed(SEED)
indices    = np.random.choice(len(img_list), IMAGES_PER_CORRUPTION, replace=False)
img_subset = [img_list[i] for i in sorted(indices)]

print(f'Total val images : {len(img_list)}')
print(f'Classes          : {len(synset_to_idx)}')
print(f'Subset size      : {len(img_subset)}')

In [ ]:
# ── Cell 7: Dataset class + loader ───────────────────────────────────────
NORMALIZE = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std =[0.229, 0.224, 0.225]
)
TO_TENSOR = transforms.Compose([transforms.ToTensor(), NORMALIZE])
RESIZE_CROP = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224)
])

class CorruptedImageNet(Dataset):
    def __init__(self, img_list, corruption_name, severity):
        self.img_list        = img_list
        self.corruption_name = corruption_name
        self.severity        = severity

    def __len__(self):
        return len(self.img_list)

    def __getitem__(self, idx):
        path, label = self.img_list[idx]
        img    = Image.open(path).convert('RGB')
        img    = RESIZE_CROP(img)
        img_np = np.array(img, dtype=np.uint8)
        # NO try/except — corruptions verified in Cell 4
        img_np = corrupt(img_np,
                         corruption_name=self.corruption_name,
                         severity=self.severity)
        return TO_TENSOR(Image.fromarray(img_np.astype(np.uint8))), label

def get_loader(corruption):
    return DataLoader(
        CorruptedImageNet(img_subset, corruption, SEVERITY),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True
    )

# Quick verify: corruptions actually change pixels
print('Verifying corruptions on subset[0]...')
p0, _ = img_subset[0]
clean = np.array(RESIZE_CROP(Image.open(p0).convert('RGB')), dtype=np.uint8)
for name in CORRUPTIONS_TO_RUN:
    result = corrupt(clean.copy(), corruption_name=name, severity=SEVERITY)
    diff   = np.abs(result.astype(int) - clean.astype(int)).mean()
    status = '[OK]' if diff > 1.0 else '[WARNING — no change]'
    print(f'  {name}: pixel_diff = {diff:.2f}  {status}')
print('Dataset ready.')

In [ ]:
# ── Cell 8: Model utilities (identical to run) ───────────────────────────
def load_resnet50():
    try:
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    except AttributeError:
        m = models.resnet50(pretrained=True)
    return m.to(device)

def configure_for_tta(model):
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.train()
            m.requires_grad_(True)
            m.track_running_stats = False
            m.running_mean = None
            m.running_var  = None
    return model

def get_bn_params(model):
    return [p for m in model.modules()
            if isinstance(m, nn.BatchNorm2d)
            for p in m.parameters() if p.requires_grad]

def save_theta0(model):
    t0 = {}
    for nm, m in model.named_modules():
        if isinstance(m, nn.BatchNorm2d):
            t0[nm + '.w'] = m.weight.data.clone()
            t0[nm + '.b'] = m.bias.data.clone()
    return t0

print('Model utilities ready.')

In [ ]:
# ── Cell 9: Loss functions + metrics (identical to run) ──────────────────
def entropy_per_sample(outputs):
    p = torch.softmax(outputs, dim=1).clamp(min=1e-8)
    return -(p * p.log()).sum(dim=1)

def mi_loss(outputs):
    p   = torch.softmax(outputs, dim=1).clamp(min=1e-8)
    hyx = -(p * p.log()).sum(dim=1).mean()
    q   = p.mean(dim=0).clamp(min=1e-8)
    hy  = -(q * q.log()).sum()
    return hyx - hy

def anchor_loss(model, theta0):
    loss = torch.tensor(0.0, device=device)
    for nm, m in model.named_modules():
        if isinstance(m, nn.BatchNorm2d):
            loss = loss + (m.weight - theta0[nm + '.w']).pow(2).sum()
            loss = loss + (m.bias   - theta0[nm + '.b']).pow(2).sum()
    return loss

def compute_metrics(all_preds, all_labels, hyx_sum, p_sum, n_total):
    preds  = torch.cat(all_preds)
    labels = torch.cat(all_labels)
    acc    = preds.eq(labels).float().mean().item() * 100.0
    counts = torch.bincount(preds, minlength=K)
    dom    = counts.max().item() / preds.shape[0]
    active = (counts > 0).sum().item()
    collapsed = int(
        (dom > COLLAPSE_DOM_THRESH and active < COLLAPSE_ACTIVE_THRESH)
        or active < COLLAPSE_HARD_THRESH
    )
    hyx = hyx_sum / n_total
    q   = (p_sum / n_total).clamp(min=1e-8)
    hy  = -(q * q.log()).sum().item()
    mi  = max(hy - hyx, 0.0)
    return dict(
        acc=round(acc, 2), collapsed=collapsed,
        dom_pct=round(dom * 100, 3), hy=round(hy, 4),
        hyx=round(hyx, 4), mi=round(mi, 4),
        active_classes=active
    )

print('Loss functions and metrics ready.')

In [ ]:
# ── Cell 10: TTA methods (identical to run) ──────────────────────────────
def run_source(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    hyx_sum, p_sum, n_total = 0.0, torch.zeros(K), 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outs   = model(images)
            p = torch.softmax(outs, dim=1).clamp(min=1e-8).cpu()
            hyx_sum += -(p * p.log()).sum(dim=1).sum().item()
            p_sum   += p.sum(dim=0)
            n_total += len(labels)
            all_preds.append(outs.argmax(1).cpu())
            all_labels.append(labels)
    return compute_metrics(all_preds, all_labels, hyx_sum, p_sum, n_total)

def run_tent(model, loader, lr):
    configure_for_tta(model)
    opt = torch.optim.SGD(get_bn_params(model), lr=lr, momentum=0.9)
    all_preds, all_labels = [], []
    hyx_sum, p_sum, n_total = 0.0, torch.zeros(K), 0
    for images, labels in loader:
        images = images.to(device)
        opt.zero_grad()
        outs = model(images)
        entropy_per_sample(outs).mean().backward()
        opt.step()
        with torch.no_grad():
            p = torch.softmax(outs, dim=1).clamp(min=1e-8).cpu().detach()
        hyx_sum += -(p * p.log()).sum(dim=1).sum().item()
        p_sum   += p.sum(dim=0)
        n_total += len(labels)
        all_preds.append(outs.argmax(1).cpu().detach())
        all_labels.append(labels)
    return compute_metrics(all_preds, all_labels, hyx_sum, p_sum, n_total)

def estimate_fisher(model, loader):
    fisher = {}
    for nm, m in model.named_modules():
        if isinstance(m, nn.BatchNorm2d):
            fisher[nm + '.w'] = torch.zeros_like(m.weight)
            fisher[nm + '.b'] = torch.zeros_like(m.bias)
    model.zero_grad()
    n_seen = 0
    for images, _ in loader:
        if n_seen >= FISHER_N: break
        images = images.to(device)
        bsz    = images.shape[0]
        entropy_per_sample(model(images)).mean().backward()
        for nm, m in model.named_modules():
            if isinstance(m, nn.BatchNorm2d):
                if m.weight.grad is not None:
                    fisher[nm + '.w'] += m.weight.grad.data.pow(2) * bsz
                if m.bias.grad is not None:
                    fisher[nm + '.b'] += m.bias.grad.data.pow(2) * bsz
        model.zero_grad()
        n_seen += bsz
    for k in fisher: fisher[k] /= max(n_seen, 1)
    return fisher

def run_eata(model, loader, lr):
    configure_for_tta(model)
    theta0 = save_theta0(model)
    fisher = estimate_fisher(model, loader)
    opt    = torch.optim.SGD(get_bn_params(model), lr=lr, momentum=0.9)
    all_preds, all_labels = [], []
    hyx_sum, p_sum, n_total = 0.0, torch.zeros(K), 0
    for images, labels in loader:
        images = images.to(device)
        opt.zero_grad()
        outs = model(images)
        ent  = entropy_per_sample(outs)
        mask = ent < E0
        if mask.sum() > 0:
            reg = torch.tensor(0.0, device=device)
            for nm, m in model.named_modules():
                if isinstance(m, nn.BatchNorm2d):
                    reg = reg + (fisher[nm+'.w'] * (m.weight - theta0[nm+'.w']).pow(2)).sum()
                    reg = reg + (fisher[nm+'.b'] * (m.bias   - theta0[nm+'.b']).pow(2)).sum()
            (ent[mask].mean() + (FISHER_WEIGHT / 2.0) * reg).backward()
            opt.step()
        with torch.no_grad():
            p = torch.softmax(outs, dim=1).clamp(min=1e-8).cpu().detach()
        hyx_sum += -(p * p.log()).sum(dim=1).sum().item()
        p_sum   += p.sum(dim=0)
        n_total += len(labels)
        all_preds.append(outs.argmax(1).cpu().detach())
        all_labels.append(labels)
    return compute_metrics(all_preds, all_labels, hyx_sum, p_sum, n_total)

def run_bmia_fixed(model, loader, lr):
    configure_for_tta(model)
    theta0 = save_theta0(model)
    opt    = torch.optim.SGD(get_bn_params(model), lr=lr, momentum=0.9)
    all_preds, all_labels = [], []
    hyx_sum, p_sum, n_total = 0.0, torch.zeros(K), 0
    for images, labels in loader:
        images = images.to(device)
        opt.zero_grad()
        outs = model(images)
        (mi_loss(outs) + LAMBDA_FIXED * anchor_loss(model, theta0)).backward()
        opt.step()
        with torch.no_grad():
            p = torch.softmax(outs, dim=1).clamp(min=1e-8).cpu().detach()
        hyx_sum += -(p * p.log()).sum(dim=1).sum().item()
        p_sum   += p.sum(dim=0)
        n_total += len(labels)
        all_preds.append(outs.argmax(1).cpu().detach())
        all_labels.append(labels)
    return compute_metrics(all_preds, all_labels, hyx_sum, p_sum, n_total)

def run_bmia_adaptive(model, loader, lr):
    configure_for_tta(model)
    theta0 = save_theta0(model)
    opt    = torch.optim.SGD(get_bn_params(model), lr=lr, momentum=0.9)
    lam    = LAMBDA_INIT
    all_preds, all_labels = [], []
    hyx_sum, p_sum, n_total = 0.0, torch.zeros(K), 0
    for images, labels in loader:
        images = images.to(device)
        opt.zero_grad()
        outs = model(images)
        (mi_loss(outs) + lam * anchor_loss(model, theta0)).backward()
        opt.step()
        with torch.no_grad():
            preds  = outs.argmax(1)
            counts = torch.bincount(preds, minlength=K)
            dom    = counts.max().item() / preds.shape[0]
        lam = min(LAMBDA_MAX, lam * 1.1) if dom > TAU else max(LAMBDA_MIN, lam * 0.95)
        with torch.no_grad():
            p = torch.softmax(outs, dim=1).clamp(min=1e-8).cpu().detach()
        hyx_sum += -(p * p.log()).sum(dim=1).sum().item()
        p_sum   += p.sum(dim=0)
        n_total += len(labels)
        all_preds.append(preds.cpu())
        all_labels.append(labels)
    return compute_metrics(all_preds, all_labels, hyx_sum, p_sum, n_total)

print('All TTA methods ready.')

In [ ]:
# ── Cell 11: Run fog + glass_blur — all methods, both LRs ────────────────
FIELDS = ['corruption','lr','method','acc','collapsed',
          'dom_pct','hy','hyx','mi','active_classes','time_s']

results_file = os.path.join(RESULTS_DIR, 'fog_glass_results.csv')
with open(results_file, 'w', newline='') as f:
    csv.DictWriter(f, fieldnames=FIELDS).writeheader()

TTA_METHODS = ['TENT', 'EATA', 'BMIA-Fixed', 'BMIA-Adaptive']
total_start = time.time()

for corruption in CORRUPTIONS_TO_RUN:
    print(f'\n{"="*65}')
    print(f'  Corruption: {corruption}')
    print(f'{"="*65}')
    loader = get_loader(corruption)

    # Source
    t0    = time.time()
    model = load_resnet50()
    src   = run_source(model, loader)
    del model
    elapsed = round(time.time() - t0, 1)
    print(f"  {'Source':20s} lr=N/A  acc={src['acc']:.1f}%  "
          f"col={src['collapsed']}  MI={src['mi']:.3f}  [{elapsed}s]")
    with open(results_file, 'a', newline='') as f:
        csv.DictWriter(f, fieldnames=FIELDS).writerow(
            dict(corruption=corruption, lr='N/A', method='Source', **src, time_s=elapsed))

    for lr in LEARNING_RATES:
        for method in TTA_METHODS:
            t0    = time.time()
            model = load_resnet50()
            if   method == 'TENT':          metrics = run_tent(model, loader, lr)
            elif method == 'EATA':          metrics = run_eata(model, loader, lr)
            elif method == 'BMIA-Fixed':    metrics = run_bmia_fixed(model, loader, lr)
            elif method == 'BMIA-Adaptive': metrics = run_bmia_adaptive(model, loader, lr)
            del model
            elapsed = round(time.time() - t0, 1)
            print(f"  {method:20s} lr={lr}  acc={metrics['acc']:.1f}%  "
                  f"col={metrics['collapsed']}  MI={metrics['mi']:.3f}  [{elapsed}s]")
            with open(results_file, 'a', newline='') as f:
                csv.DictWriter(f, fieldnames=FIELDS).writerow(
                    dict(corruption=corruption, lr=lr, method=method,
                         **metrics, time_s=elapsed))

total_time = round(time.time() - total_start)
print(f'\nDONE. Total time: {total_time}s')
print(f'Results saved to: {results_file}')

In [ ]:
# ── Cell 12: Summary ─────────────────────────────────────────────────────
import pandas as pd

df = pd.read_csv(results_file)
LOG_K = float(np.log(K))

print('='*65)
print('FOG + GLASS_BLUR RESULTS — Severity 5, 5000 images, ResNet-50')
print(f'CSD Theorem MI upper bound: log(K=1000) = {LOG_K:.3f} nats')
print('='*65)

for lr in LEARNING_RATES:
    sub = df[df['lr'] == lr]
    if sub.empty: continue
    print(f'\n── lr = {lr} ──')
    print(sub.groupby('method')[['acc','collapsed','mi']].mean().round(2).to_string())

src_df = df[df['method'] == 'Source']
print(f'\n── Source ──')
print(f"  Avg acc: {src_df['acc'].mean():.2f}%  MI: {src_df['mi'].mean():.3f}")

print('\n── Per-Corruption Accuracy (lr=0.005) ──')
sub = df[df['lr'] == 0.005]
if not sub.empty:
    pivot = sub.pivot_table(index='corruption', columns='method', values='acc')
    src   = df[df['method']=='Source'].set_index('corruption')['acc']
    pivot.insert(0, 'Source', src)
    cols = ['Source','TENT','EATA','BMIA-Fixed','BMIA-Adaptive']
    pivot = pivot[[c for c in cols if c in pivot.columns]]
    print(pivot.round(1).to_string())